In [1]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel
from torch.cuda.amp import autocast, GradScaler

In [2]:
# ------------------ CONFIG ------------------
MAX_LEN = 96
BATCH_SIZE = 16
EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ------------------ TOKENIZER ------------------
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# ------------------ LOAD DATA ------------------
data = []
with open("/content/dataset.json") as f:
    dataset = json.load(f)

for item in dataset:
    item["Distortion"] = item["Distortion"].strip().title()
    text = item["Patient Question"]
    span = item["Distorted Part"]
    label = item["Distortion"]

    if span and label:
        data.append({"text": text, "span": span, "label": label})

In [5]:
# ------------------ LABEL ENCODING ------------------
le = LabelEncoder()
labels_encoded = le.fit_transform([d["label"] for d in data])
num_labels = len(le.classes_)

In [6]:
# ------------------ FIND SPAN ------------------
def find_span(text, span):
    start = text.lower().find(span.lower())
    if start == -1:
        return -1, -1
    return start, start + len(span)

In [7]:
# ------------------ TOKENIZATION ------------------
input_ids, masks = [], []
start_positions, end_positions, labels = [], [], []

for i, item in enumerate(data):

    text = item["text"]
    span = item["span"]

    start_char, end_char = find_span(text, span)
    if start_char == -1:
        continue

    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_offsets_mapping=True
    )

    offsets = enc["offset_mapping"]

    start_tok, end_tok = None, None

    for idx, (s, e) in enumerate(offsets):
        if s <= start_char < e:
            start_tok = idx
        if s <= end_char - 1 < e:
            end_tok = idx

    if start_tok is None or end_tok is None:
        continue

    input_ids.append(enc["input_ids"])
    masks.append(enc["attention_mask"])
    start_positions.append(start_tok)
    end_positions.append(end_tok)
    labels.append(labels_encoded[i])

In [8]:
# ------------------ TENSORS ------------------
input_ids = torch.tensor(input_ids)
masks = torch.tensor(masks)
start_positions = torch.tensor(start_positions)
end_positions = torch.tensor(end_positions)
labels = torch.tensor(labels)

y_emotion = torch.tensor(np.load("y_emotion.npy"), dtype=torch.float)

In [9]:
# ------------------ DATASET ------------------
class MultiTaskDataset(Dataset):
    def __init__(self, ids, masks, labels, start, end, emotions):
        self.ids = ids
        self.masks = masks
        self.labels = labels
        self.start = start
        self.end = end
        self.emotions = emotions

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.ids[idx],
            "attention_mask": self.masks[idx],
            "labels": self.labels[idx],
            "start_positions": self.start[idx],
            "end_positions": self.end[idx],
            "emotions": self.emotions[idx]
        }

dataset = MultiTaskDataset(input_ids, masks, labels, start_positions, end_positions, y_emotion)

In [10]:
# ------------------ SPLIT ------------------
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [11]:
# ------------------ MODEL ------------------
class MultiTaskModel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained("distilroberta-base")
        hidden = self.roberta.config.hidden_size

        self.attn = nn.MultiheadAttention(hidden, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.2)

        self.emotion = nn.Linear(hidden, 28)
        self.distortion = nn.Linear(hidden, num_labels)
        self.qa = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):

        x = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        x = self.dropout(x)

        cls = x[:, 0, :]

        emotion_logits = self.emotion(cls)
        distortion_logits = self.distortion(cls)

        qa_logits = self.qa(x)

        start_logits = qa_logits[..., 0]
        end_logits = qa_logits[..., 1]

        return (
            emotion_logits.contiguous(),
            distortion_logits.contiguous(),
            start_logits.contiguous(),
            end_logits.contiguous()
        )

model = MultiTaskModel(num_labels).to(DEVICE)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# ------------------ LOSS ------------------
bce_loss = nn.BCEWithLogitsLoss()
ce_loss = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.3, patience=2, min_lr=1e-6
)

In [13]:
# ------------------ EARLY STOPPING ------------------
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.best = float("inf")
        self.counter = 0

    def step(self, loss, model):
        if loss < self.best:
            self.best = loss
            self.counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            self.counter += 1

        if self.counter >= self.patience:
            print("Early stopping triggered")
            return True
        return False

early_stopping = EarlyStopping()

In [14]:
# ------------------ AMP ------------------
scaler = GradScaler()

/tmp/ipykernel_795/552245949.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [15]:
# ------------------ TRAIN ------------------
for epoch in range(EPOCHS):

    # TRAIN
    model.train()
    train_loss = 0
    train_correct, train_total = 0, 0

    for batch in train_loader:

        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbl = batch["labels"].to(DEVICE)
        st = batch["start_positions"].to(DEVICE)
        en = batch["end_positions"].to(DEVICE)
        emo = batch["emotions"].to(DEVICE)

        optimizer.zero_grad()

        with autocast():
            e_log, d_log, s_log, e2_log = model(ids, mask)

            loss = (
                0.5 * bce_loss(e_log, emo) +
                ce_loss(d_log, lbl) +
                ce_loss(s_log, st) +
                ce_loss(e2_log, en)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        # TRAIN ACCURACY
        preds = torch.argmax(d_log, dim=1)
        train_correct += (preds == lbl).sum().item()
        train_total += lbl.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total if train_total > 0 else 0

    # VALIDATION
    model.eval()
    val_loss = 0
    val_dist_loss = 0
    correct, total = 0, 0

    with torch.no_grad():
        for batch in val_loader:

            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl = batch["labels"].to(DEVICE)
            st = batch["start_positions"].to(DEVICE)
            en = batch["end_positions"].to(DEVICE)
            emo = batch["emotions"].to(DEVICE)

            with autocast():
                e_log, d_log, s_log, e2_log = model(ids, mask)

                loss_em = bce_loss(e_log, emo)
                loss_dist = ce_loss(d_log, lbl)
                loss_st = ce_loss(s_log, st)
                loss_en = ce_loss(e2_log, en)

                loss = 0.5*loss_em + loss_dist + loss_st + loss_en

            val_loss += loss.item()
            val_dist_loss += loss_dist.item()

            preds = torch.argmax(d_log, dim=1)
            correct += (preds == lbl).sum().item()
            total += lbl.size(0)

    val_loss /= len(val_loader)
    val_dist_loss /= len(val_loader)
    val_acc = correct / total if total > 0 else 0

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Distortion Accuracy: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Distortion Loss: {val_dist_loss:.4f}")
    print(f"Val Distortion Accuracy: {val_acc:.4f}")

    # LR Scheduler + Logging
    prev_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_dist_loss)
    new_lr = optimizer.param_groups[0]['lr']

    if new_lr != prev_lr:
        print(f"🔻 LR reduced: {prev_lr} → {new_lr}")

    print(f"Current LR: {new_lr}")

    # Early Stopping
    if early_stopping.step(val_dist_loss, model):
        break

/tmp/ipykernel_795/684501140.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_795/684501140.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 1
Train Loss: 3.0895
Train Distortion Accuracy: 0.6449
Val Loss: 2.1197
Val Distortion Loss: 0.6646
Val Distortion Accuracy: 0.7922
Current LR: 2e-05

Epoch 2
Train Loss: 1.9447
Train Distortion Accuracy: 0.8111
Val Loss: 1.9639
Val Distortion Loss: 0.6112
Val Distortion Accuracy: 0.8048
Current LR: 2e-05

Epoch 3
Train Loss: 1.6977
Train Distortion Accuracy: 0.8312
Val Loss: 1.9432
Val Distortion Loss: 0.6135
Val Distortion Accuracy: 0.8142
Current LR: 2e-05

Epoch 4
Train Loss: 1.5085
Train Distortion Accuracy: 0.8496
Val Loss: 2.0125
Val Distortion Loss: 0.6281
Val Distortion Accuracy: 0.8123
Current LR: 2e-05

Epoch 5
Train Loss: 1.3461
Train Distortion Accuracy: 0.8671
Val Loss: 1.9367
Val Distortion Loss: 0.5918
Val Distortion Accuracy: 0.8205
Current LR: 2e-05

Epoch 6
Train Loss: 1.2098
Train Distortion Accuracy: 0.8818
Val Loss: 2.1198
Val Distortion Loss: 0.6442
Val Distortion Accuracy: 0.8217
Current LR: 2e-05

Epoch 7
Train Loss: 1.0862
Train Distortion Accuracy: 0.8

In [16]:
# ------------------ SAVE ------------------ t
os.makedirs("model", exist_ok=True)

# Save model weights
torch.save(model.state_dict(), "model/multitask_emotion_distortion_span.pth")

# Save tokenizer
tokenizer.save_pretrained("model")

# Save label encodings
np.save("model/label_classes.npy", le.classes_)

# Emotion labels
emotion_labels = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization",
    "relief","remorse","sadness","surprise","neutral"
]

np.save("model/emotion_labels.npy", emotion_labels)

# Save config
config = {
    "max_len": 128,
    "num_labels": len(le.classes_),
    "emotion_classes": len(emotion_labels)
}

with open("model/config.json", "w") as f:
    json.dump(config, f)

print("Model saved successfully")

Model saved successfully


In [17]:
!zip -r multitask_model.zip model

  adding: model/ (stored 0%)
  adding: model/tokenizer.json (deflated 82%)
  adding: model/emotion_labels.npy (deflated 79%)
  adding: model/tokenizer_config.json (deflated 50%)
  adding: model/config.json (deflated 9%)
  adding: model/multitask_emotion_distortion_span.pth (deflated 7%)
  adding: model/label_classes.npy (deflated 76%)
